<a href="https://colab.research.google.com/github/bhagyaraju12345-svg/used_cars_data_prep.ipynb/blob/main/used_cars_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1 — Load, Inspect, and Clean**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [4]:
df = pd.read_csv("used_cars_messy.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,2581,Renault Duster,Renault,Duster,7,67647,Dealer,Diesel,Manual,19.87,1461.0,83.80,5.0,575000.0
1,224,Maruti Wagon R,Maruti,Wagon R,3,52000,Dealer,CNG,Manual,33.54,998.0,67.04,5.0,435000.0
2,1803,Honda City,Honda,City,3,48959,Dealer,Petrol,Manual,NaN,1497.0,117.60,5.0,875000.0
3,2728,Mahindra Scorpio,Mahindra,Scorpio,6,82000,dealer,Diesel,Manual,15.4,2179.0,NaN,8.0,925000.0
4,14160,Mahindra XUV500,Mahindra,XUV500,4,34000,Dealer,Diesel,Automatic,16.0,2179.0,140.00,7.0,1175000.0


In [5]:
df.shape

(15661, 14)

In [6]:
df.isnull().sum()*100/len(df)

,0
Unnamed: 0,0.000000
car_name,0.000000
brand,0.000000
model,0.000000
vehicle_age,0.000000
km_driven,0.000000
seller_type,0.000000
fuel_type,0.000000
transmission_type,0.000000
mileage,8.205095


In [7]:
# Drop rows where selling_price is missing
df = df.dropna(subset=['selling_price'])

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15503 entries, 0 to 15660
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15503 non-null  int64  
 1   car_name           15503 non-null  object 
 2   brand              15503 non-null  object 
 3   model              15503 non-null  object 
 4   vehicle_age        15503 non-null  int64  
 5   km_driven          15503 non-null  object 
 6   seller_type        15503 non-null  object 
 7   fuel_type          15503 non-null  object 
 8   transmission_type  15503 non-null  object 
 9   mileage            14234 non-null  object 
 10  engine             13651 non-null  float64
 11  max_power          14712 non-null  float64
 12  seats              15065 non-null  float64
 13  selling_price      15503 non-null  float64
dtypes: float64(4), int64(2), object(8)
memory usage: 1.8+ MB


In [9]:
df['mileage'] = df['mileage'].astype(str).str.replace('kmpl', '').str.replace('kmp', '').str.strip()

df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')
df['mileage'].head()

,mileage
0,19.87
1,33.54
2,NaN
3,15.40
4,16.00


In [10]:
for col in ['engine', 'mileage', 'max_power']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} missing values with median: {median_val}")

Filled engine missing values with median: 1248.0
Filled mileage missing values with median: 19.67
Filled max_power missing values with median: 88.5


In [12]:
df.isnull().sum() * 100/ df.shape[0]

,0
Unnamed: 0,0.00000
car_name,0.00000
brand,0.00000
model,0.00000
vehicle_age,0.00000
km_driven,0.00000
seller_type,0.00000
fuel_type,0.00000
transmission_type,0.00000
mileage,0.00000


In [13]:
# Remove clearly erroneous prices
before = len(df)

# Remove the 999999999 entries (data entry errors)
df = df[df['selling_price'] != 999999999]

# Remove suspiciously low prices (< ₹10,000 for a car is not realistic)
df = df[df['selling_price'] >= 10000]

# Also fix seats = 0 (impossible)
df = df[df['seats'] > 0]

after = len(df)
print(f"Removed {before - after} rows with impossible values")
print(f"Remaining rows: {after}")

Removed 448 rows with impossible values
Remaining rows: 15055


In [14]:
df.describe()

,Unnamed: 0,vehicle_age,mileage,engine,max_power,seats,selling_price
count,15055.000000,15055.000000,15055.000000,15055.000000,15055.000000,15055.000000,1.505500e+04
mean,9808.624045,6.038459,19.695789,1451.779077,99.961830,5.324942,7.755753e+05
std,5642.356521,3.013926,4.007502,485.517834,41.978003,0.803886,9.010542e+05
min,0.000000,0.000000,4.000000,793.000000,38.400000,2.000000,4.000000e+04
25%,4907.500000,4.000000,17.090000,1197.000000,74.000000,5.000000,3.850000e+05
50%,9872.000000,6.000000,19.670000,1248.000000,88.500000,5.000000,5.550000e+05
75%,14651.000000,8.000000,22.320000,1498.000000,113.400000,5.000000,8.200000e+05
max,19543.000000,29.000000,33.540000,6592.000000,626.000000,9.000000,3.950000e+07


In [15]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [16]:
df.shape

(14888, 14)

**Task 2 — Encode Categorical Features**

In [17]:
df['transmission_type'].value_counts()

,count
transmission_type,
Manual,11794
Automatic,3094


In [25]:
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic' : 1})
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,transmission_type,mileage,engine,max_power,...,fuel_type_petrol,fuel_type_petrol,seller_type_Dealer,seller_type_Delaer,seller_type_Individual,seller_type_Individuall,seller_type_Individul,seller_type_Trustmark Dealer,seller_type_dealer,seller_type_individual
0,2581,Renault Duster,Renault,Duster,7,67647,NaN,19.87,1461.0,83.80,...,False,False,True,False,False,False,False,False,False,False
1,224,Maruti Wagon R,Maruti,Wagon R,3,52000,NaN,33.54,998.0,67.04,...,False,False,True,False,False,False,False,False,False,False
2,1803,Honda City,Honda,City,3,48959,NaN,19.67,1497.0,117.60,...,False,False,True,False,False,False,False,False,False,False
3,2728,Mahindra Scorpio,Mahindra,Scorpio,6,82000,NaN,15.40,2179.0,88.50,...,False,False,False,False,False,False,False,False,True,False
4,14160,Mahindra XUV500,Mahindra,XUV500,4,34000,NaN,16.00,2179.0,140.00,...,False,False,True,False,False,False,False,False,False,False


In [28]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,transmission_type,mileage,engine,max_power,...,fuel_type_petrol,fuel_type_petrol,seller_type_Dealer,seller_type_Delaer,seller_type_Individual,seller_type_Individuall,seller_type_Individul,seller_type_Trustmark Dealer,seller_type_dealer,seller_type_individual
0,2581,Renault Duster,Renault,Duster,7,67647,NaN,19.87,1461.0,83.80,...,False,False,True,False,False,False,False,False,False,False
1,224,Maruti Wagon R,Maruti,Wagon R,3,52000,NaN,33.54,998.0,67.04,...,False,False,True,False,False,False,False,False,False,False
2,1803,Honda City,Honda,City,3,48959,NaN,19.67,1497.0,117.60,...,False,False,True,False,False,False,False,False,False,False
3,2728,Mahindra Scorpio,Mahindra,Scorpio,6,82000,NaN,15.40,2179.0,88.50,...,False,False,False,False,False,False,False,False,True,False
4,14160,Mahindra XUV500,Mahindra,XUV500,4,34000,NaN,16.00,2179.0,140.00,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15656,6581,Maruti Ertiga,Maruti,Ertiga,7,127731,NaN,20.77,1248.0,88.80,...,False,False,True,False,False,False,False,False,False,False
15657,17029,Volkswagen Vento,Volkswagen,Vento,11,59000,NaN,16.09,1248.0,103.20,...,False,False,True,False,False,False,False,False,False,False
15658,6839,Maruti Wagon R,Maruti,Wagon R,7,20000,NaN,20.51,998.0,67.04,...,False,False,False,False,True,False,False,False,False,False
15659,1104,Hyundai i20,Hyundai,i20,2,15000,NaN,18.60,1197.0,81.86,...,False,False,True,False,False,False,False,False,False,False


In [29]:
df.columns.tolist()

['Unnamed: 0',
 'car_name',
 'brand',
 'model',
 'vehicle_age',
 'km_driven',
 'transmission_type',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price',
 'fuel_type_ Petrol',
 'fuel_type_CNG',
 'fuel_type_DIESEL',
 'fuel_type_Diesel',
 'fuel_type_Electric',
 'fuel_type_LPG',
 'fuel_type_PETROL',
 'fuel_type_Petrol',
 'fuel_type_Petrol ',
 'fuel_type_diesel',
 'fuel_type_diesel ',
 'fuel_type_petrol',
 'fuel_type_petrol ',
 'seller_type_Dealer',
 'seller_type_Delaer',
 'seller_type_Individual',
 'seller_type_Individuall',
 'seller_type_Individul',
 'seller_type_Trustmark Dealer',
 'seller_type_dealer',
 'seller_type_individual']

Markdown Answer (write this in a markdown cell)

Why drop_first=True is used:

It avoids dummy variable trap (multicollinearity)
One category is dropped because it can be inferred from others

What does a row of all zeros mean:

It represents the dropped base category
Example: If Petrol is dropped, all zeros = Petrol

**Task 3 — Split and Baseline MAE**

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Define X and y
X = df.drop('selling_price', axis=1)
y = df['selling_price']

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline prediction (mean)
baseline_pred = [y_train.mean()] * len(y_test)

# MAE
mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹474938
